# 🎙️ EmotionSense AI — Real-Data Validation (Kaggle)

Runs the frozen benchmark on **RAVDESS** + **CREMA-D**: baselines + classical models
(speaker-independent 5-fold CV + cross-corpus), and the **frozen Distil-HuBERT + SVM**
transformer model — all under the identical protocol.

## Before you run
1. **Add Data** (right panel) → attach these public datasets:
   - `uwrfkaggler/ravdess-emotional-speech-audio`
   - `ejlok1/cremad`
2. For the transformer cell: **Settings → Accelerator → GPU**.
3. Settings → **Internet: ON** (needed to clone + pip install).

## 1. Clone the repo & confirm the Transformer Phase 1 commit is present

In [ ]:
import os, sys, subprocess

REPO = '/kaggle/working/EmotionSenseAI'
if not os.path.isdir(REPO):
    subprocess.run(
        ['git', 'clone', 'https://github.com/shashi05shankar/EmotionSenseAI.git', REPO],
        check=True,
    )
os.chdir(REPO)
subprocess.run(['git', 'pull', '--ff-only'], check=False)  # get latest if already cloned
print('cwd:', os.getcwd())
# You should see 'Transformer Phase 1' in this log. If not, push that commit to GitHub
# main from your machine, then re-run this cell before the transformer step.
subprocess.run(['git', 'log', '--oneline', '-4'])

## 2. Install dependencies & make the package importable

On Kaggle the running kernel does not pick up the editable-install path hook, and
`!python` subprocesses need it too — so we add `src/` to both `sys.path` and `PYTHONPATH`.

In [ ]:
# Core deps only (enough for the classical benchmark). transformers is added later.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

SRC = f'{REPO}/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')

import emotionsense
print('emotionsense', emotionsense.__version__, '->', emotionsense.__file__)

## 3. Link the attached datasets into `data/raw/`

RAVDESS points at the inner `audio_speech_actors_01-24` folder on purpose — the dataset
also has a duplicate top-level `Actor_*` tree, and symlinking the parent would make the
loader count every clip twice. Expected counts: **ravdess 1440**, **crema_d 7442**.

In [ ]:
import shutil
from pathlib import Path

SOURCES = {
    'ravdess': '/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio/audio_speech_actors_01-24',
    'crema_d': '/kaggle/input/datasets/ejlok1/cremad',
}

raw = Path('data/raw')
raw.mkdir(parents=True, exist_ok=True)
for name, src in SOURCES.items():
    dst = raw / name
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)
    if not os.path.isdir(src):
        print(f'❌ {name}: source not found -> {src}  (is the dataset attached?)')
        continue
    os.symlink(src, dst)
    n = len(list(dst.rglob('*.wav')))
    print(f'✅ {name} -> {src}  ({n} wav)')

## 4. Classical validation — RAVDESS 5-fold CV + cross-corpus CREMA-D

In [ ]:
!python scripts/run_benchmark.py \
    --experiment configs/experiments/rc1_validation.yaml \
    --out-dir experiments/reports

## 5. CREMA-D in-corpus 5-fold CV (91 speakers)

In [ ]:
!python scripts/run_benchmark.py \
    --experiment configs/experiments/rc1_cremad.yaml \
    --out-dir experiments/reports

## 6. Show the classical leaderboards + confusion matrices

In [ ]:
import glob

for md_file in sorted(glob.glob('experiments/reports/*.md')):
    print('=' * 80)
    print(md_file, '\n')
    print(open(md_file, encoding='utf-8').read())

## 7. Transformer Phase 1 — frozen Distil-HuBERT + SVM

Needs a **GPU** runtime and the Transformer Phase 1 commit on `main`. The SSL extractor
auto-selects CUDA; embeddings are cached per clip. The CREMA-D cross-corpus pass
(~7.4k clips) dominates runtime.

> If this errors with `Unknown classical family: distilhubert`, your GitHub `main` is
> missing the fix — push it, re-run cell 1 (git pull), then this cell.

In [ ]:
# torch is already installed on Kaggle GPU images; we only need transformers.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'], check=True)
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
!python scripts/run_benchmark.py \
    --experiment configs/experiments/rc1_transformer.yaml \
    --out-dir experiments/reports

## 8. Final results (all experiments) + JSON export

In [ ]:
import glob, json

for md_file in sorted(glob.glob('experiments/reports/*.md')):
    print('=' * 80)
    print(md_file, '\n')
    print(open(md_file, encoding='utf-8').read())

print('\nJSON reports (copy these back as the validation record):')
for j in sorted(glob.glob('experiments/reports/*.json')):
    print(' -', j)